In [ ]:
from __future__ import annotations
import numpy as np
import librosa
import soundfile as sf
from scipy import signal
from typing import Tuple

In [ ]:
def load_audio(path: str, target_sr: int = 16000) -> Tuple[np.ndarray, int]:
    y, sr = sf.read(path, always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
        sr = target_sr
    y = np.nan_to_num(y)
    return y, sr

def save_audio(path: str, y: np.ndarray, sr: int = 16000):
    sf.write(path, y, sr)

def bandpass(x: np.ndarray, sr: int, lo: float, hi: float) -> np.ndarray:
    sos = signal.butter(6, [lo, hi], btype="bandpass", fs=sr, output="sos")
    return signal.sosfilt(sos, x).astype(np.float32)

def watermark_embed(
    y: np.ndarray,
    sr: int = 16000,
    seed: int = 1337,
    strength: float = 0.004,
    band_lo: float = 6000.0,
    band_hi: float = 8000.0,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal(len(y)).astype(np.float32)
    wm = bandpass(noise, sr, band_lo, band_hi)
    wm = wm / (np.max(np.abs(wm)) + 1e-9)
    y_out = y + strength * wm
    y_out = np.clip(y_out, -1.0, 1.0)
    return y_out.astype(np.float32)

def watermark_detect(
    y: np.ndarray,
    sr: int = 16000,
    seed: int = 1337,
    band_lo: float = 6000.0,
    band_hi: float = 8000.0,
) -> float:
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal(len(y)).astype(np.float32)
    ref = bandpass(noise, sr, band_lo, band_hi)
    ref = ref / (np.linalg.norm(ref) + 1e-9)

    yy = bandpass(y.astype(np.float32), sr, band_lo, band_hi)
    yy = yy / (np.linalg.norm(yy) + 1e-9)

    score = float(np.dot(ref, yy))
    return score